In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt, colors, patches

from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import CCA


sys.path.append(os.path.join(os.getcwd(), '../../'))
from utils import read_utils
from utils import plot_utils
from utils import train_utils

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)

In [ ]:
read_ids = list(range(0, 12+1))
# read_ids = [0, 2, 3, 8, 9]
plot_id = 0

dims = 3
dims_preca = 5

selection_n = 5
selection_bhl = ['still', 'walk', 'front-groom', 'back-groom', 'abdomen-move']

In [ ]:
path = os.path.join(read_utils.get_data_path(), 'fly-2023-schaffer/brain')

(file_ids, filenames,
 traces,
 states, markers) = read_utils.get_data(path, read_ids)

# # clear nan values
# for i in range(len(file_ids)):
# 	mask = markers[i].notna().all(axis=1)
#
# 	traces[i] = traces[i][mask]
# 	markers[i] = markers[i][mask]
# 	states[i] = states[i][mask]
#
# 	#filenames[i] = filenames[i][mask]

# # remove unused discrete state labels
# states = [ds[selection_bhl] for ds in states]

# naive ID unknown neurons
for i in range(len(file_ids)):
	traces[i].columns = pd.Index([
		f'{file_ids[i]:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces[i].columns, traces[i].columns == ''))
	])

In [ ]:
# prep
def highpass_butter(x, fs, cutoff_hz, order=4, axis=0):
    return sp.signal.sosfiltfilt(sp.signal.butter(order, cutoff_hz, btype='highpass', fs=fs, output='sos'), x, axis=axis)

sampling_rates = [read_utils.get_sampling_rate(os.path.join(path, filenames[file_ids.index(file_id)])) for file_id in file_ids]

traces = [pd.DataFrame(data=trace.values[1:, :], columns=trace.columns) for trace in traces]
traces = [pd.DataFrame(highpass_butter(trace.values, sr, 1 / 100), columns=trace.columns) for trace, sr in zip(traces, sampling_rates)]

states = [pd.DataFrame(data=state.values[1:, :], columns=state.columns) for state in states]

markers = [pd.DataFrame(data=marker.values[1:, :] - marker.values[:-1, :], columns=marker.columns) for marker in markers]
markers = [
	pd.DataFrame(
		data=(marker.values - marker.values.mean(axis=0)) / np.where((marker_std := np.std(marker.values, axis=0)) == 0.0, 1.0, marker_std),
		columns=marker.columns)
	for marker in markers
]

# man

In [ ]:
(modes_manifold, modes_manifold_b,
 recos_manifold, recos_manifold_b,
 loads_cca_man, loads_cca_man_b,
 vars_expl_mans, vars_expl_mans_rs) = train_utils.get_manifolds_cca(traces,
																	[ds[selection_bhl] for ds in states],
																	n_cv=dims, n_pc=dims_preca)

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
						  modes_manifold, states,
						  [vx[:2] for vx in vars_expl_mans], [vx[:2] for vx in vars_expl_mans_rs] if dims_preca > 0 else None,
						  title='CCA manifolds', top_space=2.5, save_name=None)  #f'{f'{dims_preca}-dim-PCA_' if dims_preca > 0 else ''}{dims}-dim-CCA'

In [ ]:
# plot_utils.plot_manifolds_3d([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
# 							 modes_manifold, [ds[selection_bhl] for ds in states],
# 							 [vx[:3] for vx in vars_expl_mans], [vx[:3] for vx in vars_expl_mans_rs] if dims_preca > 0 else None,
# 							 title='CCA manifolds', save_name=None)  #f'{f'{dims_preca}-dim-PCA_' if dims_preca > 0 else ''}{dims}-dim-CCA_3d'

In [ ]:
for file_id in file_ids:
	corrs = [
		np.corrcoef(modes_manifold[file_ids.index(file_id)].values[:, i], modes_manifold_b[file_ids.index(file_id)].values[:, i])[0, 1] for i in range(modes_manifold[file_ids.index(file_id)].values.shape[1])
	]
	print(f'{file_id}: {filenames[file_ids.index(file_id)]}')
	for i, corr in enumerate(corrs):
		print(f'\tCV{i}: {corr:.2f}')

In [ ]:
raise Exception

# residuals

In [ ]:
markers_zs = [pd.DataFrame(data=StandardScaler().fit_transform(marker.values), columns=marker.columns) for marker in markers]

In [ ]:
traces_residual = [pd.DataFrame(data=trace - reco_man, columns=trace.columns) for trace, reco_man in zip(traces, recos_manifold)]

In [ ]:
# for t_res in traces_residual:
# 	print(np.median(t_res.values, axis=0))

In [ ]:
n_cv = markers_zs[file_ids.index(plot_id)].values.shape[1]

(modes_concrete, modes_concrete_b,
 recos_concrete, recos_concrete_b,
 loads_cca_con, loads_cca_con_b,
 vars_expl_cons, vars_expl_cons_rs) = train_utils.get_manifolds_cca(traces_residual,
																	markers_zs,
																	n_cv=n_cv, n_pc=0)

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
						  modes_concrete, markers_zs,
						  [vx[:2] for vx in vars_expl_cons], None,
						  title='CCA concretes', save_name=None, top_space=6)

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
						  modes_concrete_b, markers_zs,
						  [vx[:2] for vx in vars_expl_cons], None,
						  title='CCA concretes b', save_name=None, top_space=7)

In [ ]:
for file_id in file_ids:
	corrs = [
		np.corrcoef(modes_concrete[file_ids.index(file_id)].values[:, i], modes_concrete_b[file_ids.index(file_id)].values[:, i])[0, 1] for i in range(modes_concrete[file_ids.index(file_id)].values.shape[1])
	]
	print(f'{filenames[file_ids.index(file_id)]}')
	for i, corr in enumerate(corrs):
		print(f'\tCV{i}: {corr:.2f}')